In [9]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 1 - Week 8 Bayesian Optimisation
# --------------------------------------------------
#
# Method:
# - maximise the raw objective
# - fit an ARD Matern GP to all accumulated observations
# - optimise GP hyperparameters from the data
# - use fitted ARD lengthscales to guide candidate generation
# - use Expected Improvement as the primary acquisition
# - compare UCB only as a diagnostic
#
# No manual centring on previous weeks.
# No closeness-to-zero transformation.
# No artificial uncertainty inflation.

In [10]:
X = np.load("function1/initial_inputs.npy")
Y = np.load("function1/initial_outputs.npy").reshape(-1)

assert X.shape[1] == 2
assert len(X) == len(Y)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("\nCurrent best observed input:")
print(best_x)

print("\nCurrent best observed output:")
print(best_y)

print("\nObserved Y range:")
print("min =", np.min(Y))
print("max =", np.max(Y))
print("std =", np.std(Y))

X shape: (17, 2)
Y shape: (17,)

Current best observed input:
[0.73102363 0.73299988]

Current best observed output:
7.710875114502849e-16

Observed Y range:
min = -0.0036060626443634764
max = 7.710875114502849e-16
std = 0.0008484853280855351


In [11]:
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(2) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
1.06**2 * Matern(length_scale=[2, 0.0332], nu=2.5) + WhiteKernel(noise_level=1e-08)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [12]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


ARD lengthscales:
[2.         0.03321078]

Normalised inverse-lengthscale sensitivity:
[0.01633416 0.98366584]


In [13]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.02,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.05,
    0.20
)

print("Local widths:", local_scale)
print("Wide widths:", wide_scale)

Local widths: [0.1  0.02]
Wide widths: [0.2  0.05]


In [14]:
rng = np.random.default_rng(42)

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(30000, 2)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(20000, 2)
    )
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

In [15]:
grid_axis = np.linspace(
    0,
    1,
    401
)

x1_grid, x2_grid = np.meshgrid(
    grid_axis,
    grid_axis
)

global_candidates = np.column_stack([
    x1_grid.ravel(),
    x2_grid.ravel()
])

print(
    "Global grid candidates:",
    len(global_candidates)
)

Global grid candidates: 160801


In [16]:
candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

print(
    "Total candidates before filtering:",
    len(candidates)
)

Total candidates before filtering: 210801


In [17]:
tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "Candidates after filtering:",
    len(candidates)
)

Candidates after filtering: 208529


In [18]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

print("Prediction complete.")
print("Mean predicted std:", np.mean(sigma))

Prediction complete.
Mean predicted std: 0.0004602054758226597


In [19]:
def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    Z = np.zeros_like(mu)

    valid = sigma > 1e-15

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI

In [20]:
EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\nPRIMARY ACQUISITION: Expected Improvement")

print("candidate =", candidates[ei_idx])
print("predicted mean =", mu[ei_idx])
print("predicted std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


PRIMARY ACQUISITION: Expected Improvement
candidate = [0.905 0.535]
predicted mean = -0.00011570040125237232
predicted std = 0.0008517870723878745
EI = 0.0002850937297182466


In [21]:
y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\nEI sensitivity check:\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", f"{mu[idx]:.6e}",
        "\n std =", f"{sigma[idx]:.6e}",
        "\n EI =", f"{EI_test[idx]:.6e}",
        "\n"
    )


EI sensitivity check:

xi = 0.000000e+00 
 candidate = [0.905 0.535] 
 mean = -1.157004e-04 
 std = 8.517871e-04 
 EI = 2.850937e-04 

xi = 8.484853e-06 
 candidate = [0.9075 0.535 ] 
 mean = -1.157015e-04 
 std = 8.517883e-04 
 EI = 2.813264e-04 

xi = 4.242427e-05 
 candidate = [0.9125 0.535 ] 
 mean = -1.157041e-04 
 std = 8.517912e-04 
 EI = 2.665901e-04 

xi = 8.484853e-05 
 candidate = [0.8925 0.5325] 
 mean = -1.233038e-04 
 std = 8.600150e-04 
 EI = 2.490207e-04 



In [22]:
print("\nUCB diagnostic:\n")

for beta in [
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", f"{mu[idx]:.6e}",
        "\n std =", f"{sigma[idx]:.6e}",
        "\n UCB =", f"{UCB[idx]:.6e}",
        "\n"
    )


UCB diagnostic:

beta=0.1 
 candidate = [1.         0.74071304] 
 mean = 1.005299e-04 
 std = 1.776377e-04 
 UCB = 1.182936e-04 

beta=0.25 
 candidate = [1.         0.74103727] 
 mean = 1.002909e-04 
 std = 1.789888e-04 
 UCB = 1.450381e-04 

beta=0.5 
 candidate = [0.945  0.5475] 
 mean = -6.848999e-05 
 std = 7.785960e-04 
 UCB = 3.208080e-04 

beta=1.0 
 candidate = [0.895  0.5325] 
 mean = -1.233042e-04 
 std = 8.600155e-04 
 UCB = 7.367112e-04 



In [23]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "mean =",
    mu[mean_idx]
)

print(
    "std =",
    sigma[mean_idx]
)


Highest predicted mean:
candidate = [1.         0.74050503]
mean = 0.00010058071094020703
std = 0.0001766825585042735


In [24]:
# --------------------------------------------------
# Function 1 Week 8 final recommendation
# --------------------------------------------------
#
# Standard Expected Improvement is the declared primary
# acquisition rule for Week 8.
#
# The candidate is selected automatically from the GP
# posterior; no candidate is manually inserted.

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

final_idx = np.argmax(EI)

week8_candidate = candidates[final_idx]

print("Week 8 Function 1 candidate:")
print(week8_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nExpected Improvement:")
print(EI[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week8_candidate
)

print("\nPortal format:")
print(portal)

Week 8 Function 1 candidate:
[0.905 0.535]

Predicted mean:
-0.00011570040125237232

Predicted std:
0.0008517870723878745

Expected Improvement:
0.0002850937297182466

Portal format:
0.905000-0.535000


In [25]:
# --------------------------------------------------
# Function 1 - ARD trust-region search
# --------------------------------------------------
#
# Global acquisition diagnostics are unstable:
# - EI is dominated by high uncertainty around x2 ~ 0.53
# - the GP mean extrapolates very strongly towards x1 = 1
#
# Therefore use a formal trust region centred automatically
# on the best observed point.
#
# The widths are determined from the fitted ARD lengthscales,
# not manually chosen around a previous week's query.

trust_half_width = np.clip(
    0.5 * lengthscales,
    0.025,
    0.15
)

lower = np.maximum(
    0.0,
    best_x - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x + trust_half_width
)

print("Current best:", best_x)
print("ARD lengthscales:", lengthscales)
print("Trust-region half widths:", trust_half_width)
print("Lower bounds:", lower)
print("Upper bounds:", upper)

Current best: [0.73102363 0.73299988]
ARD lengthscales: [2.         0.03321078]
Trust-region half widths: [0.15  0.025]
Lower bounds: [0.58102363 0.70799988]
Upper bounds: [0.88102363 0.75799988]


In [26]:
# Dense 2D trust-region grid

x1_axis = np.linspace(
    lower[0],
    upper[0],
    501
)

x2_axis = np.linspace(
    lower[1],
    upper[1],
    501
)

g1, g2 = np.meshgrid(
    x1_axis,
    x2_axis
)

tr_candidates = np.column_stack([
    g1.ravel(),
    g2.ravel()
])

# Remove existing / near-duplicate evaluations
tree = cKDTree(X)

distance, _ = tree.query(
    tr_candidates,
    k=1
)

tr_candidates = tr_candidates[
    distance > 0.01
]

print(
    "Trust-region candidates:",
    len(tr_candidates)
)

Trust-region candidates: 243240


In [27]:
tr_mu, tr_sigma = gp.predict(
    tr_candidates,
    return_std=True
)

In [28]:
tr_EI = expected_improvement(
    tr_mu,
    tr_sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(tr_EI)

print("\nTrust-region EI:")
print("candidate =", tr_candidates[ei_idx])
print("mean =", tr_mu[ei_idx])
print("std =", tr_sigma[ei_idx])
print("EI =", tr_EI[ei_idx])


Trust-region EI:
candidate = [0.88102363 0.74149988]
mean = 8.902903800467573e-05
std = 0.00014631037595458956
EI = 0.00011336851960052393


In [29]:
# Highest mean inside trust region

mean_idx = np.argmax(tr_mu)

print("\nTrust-region highest predicted mean:")
print("candidate =", tr_candidates[mean_idx])
print("mean =", tr_mu[mean_idx])
print("std =", tr_sigma[mean_idx])


# UCB diagnostics inside trust region

print("\nTrust-region UCB:\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = tr_mu + beta * tr_sigma

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", tr_candidates[idx],
        "\n mean =", f"{tr_mu[idx]:.6e}",
        "\n std =", f"{tr_sigma[idx]:.6e}",
        "\n UCB =", f"{UCB[idx]:.6e}",
        "\n"
    )


Trust-region highest predicted mean:
candidate = [0.88102363 0.74039988]
mean = 9.019172440381771e-05
std = 0.00014063330128790617

Trust-region UCB:

beta=0.1 
 candidate = [0.88102363 0.74069988] 
 mean = 9.008513e-05 
 std = 1.423920e-04 
 UCB = 1.043243e-04 

beta=0.25 
 candidate = [0.88102363 0.74109988] 
 mean = 8.969175e-05 
 std = 1.444916e-04 
 UCB = 1.258146e-04 

beta=0.5 
 candidate = [0.88102363 0.74149988] 
 mean = 8.902904e-05 
 std = 1.463104e-04 
 UCB = 1.621842e-04 

beta=1.0 
 candidate = [0.88102363 0.74219988] 
 mean = 8.727870e-05 
 std = 1.488187e-04 
 UCB = 2.360974e-04 



In [30]:
# --------------------------------------------------
# Final Function 1 Week 8 selection
# --------------------------------------------------
#
# Global EI was dominated by predictive uncertainty and selected
# a lower-mean region around x2 ~ 0.535.
#
# Because Function 1 showed unstable long-range GP extrapolation,
# the acquisition was re-optimised inside an incumbent-centred
# trust region whose widths were determined from the fitted ARD
# lengthscales.
#
# Within this trust region:
# - Expected Improvement selected [0.881024, 0.741500]
# - UCB beta=0.5 selected the same point
# - the highest GP mean occurred very nearby
#
# EI remains the declared primary acquisition, so its maximum
# inside the trust region is used automatically as the Week 8 query.

tr_EI = expected_improvement(
    tr_mu,
    tr_sigma,
    best_y,
    xi=0.0
)

final_idx = np.argmax(tr_EI)

week8_candidate = tr_candidates[final_idx]

print("Week 8 Function 1 candidate:")
print(week8_candidate)

print("\nPredicted mean:")
print(tr_mu[final_idx])

print("\nPredicted std:")
print(tr_sigma[final_idx])

print("\nExpected Improvement:")
print(tr_EI[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week8_candidate
)

print("\nPortal format:")
print(portal)

Week 8 Function 1 candidate:
[0.88102363 0.74149988]

Predicted mean:
8.902903800467573e-05

Predicted std:
0.00014631037595458956

Expected Improvement:
0.00011336851960052393

Portal format:
0.881024-0.741500
